# Probability and Bayes' Theorem for Machine Learning
How machines reason about uncertainty, explained simply. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. Probability basics
A **probability** is a number from 0 (impossible) to 1 (certain): favourable outcomes divided by all equally likely outcomes. We can also estimate it by repeating an experiment many times and counting.

### Flipping coins
- `rng.integers(0, 2, size=n)`: n random 0s and 1s: a coin flip each
- `flips.mean()`: the share of 1s = the share of heads
- `the pattern`: with more flips, the share gets closer to 0.5 (the law of large numbers)

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
for n in [10, 1000, 100000]:
    flips = rng.integers(0, 2, size=n)   # 1 = heads
    print(n, "flips:", flips.mean())

### Rolling a die
- `rolls == 6`: True for every six; .mean() gives the share: about 1/6 = 0.167
- `rolls % 2 == 0`: even numbers (remainder 0 when divided by 2): 3 of 6 faces, about 0.5
- `rolls != 6`: the complement: 1 - 1/6, about 0.833

In [ ]:
rolls = rng.integers(1, 7, size=60000)
print((rolls == 6).mean())          # P(six)
print((rolls % 2 == 0).mean())      # P(even)
print((rolls != 6).mean())          # P(not six)

### Watching the estimate settle
- `np.cumsum(...)`: a running count of sixes after each roll
- `/ np.arange(1, 2001)`: divide by how many rolls so far: the running share
- `plt.axhline(1/6)`: the true probability, as a dashed line

In [ ]:
import matplotlib.pyplot as plt

share = np.cumsum(rolls[:2000] == 6) / np.arange(1, 2001)
plt.plot(share)
plt.axhline(1/6, color="red", linestyle="--", label="1/6")
plt.legend()
plt.title("Share of sixes as we keep rolling")
plt.xlabel("number of rolls")
plt.ylabel("share of sixes")
plt.show()

### AND and OR with two dice
- `a, b`: 100,000 rolls of two separate dice
- `&`: both are six: independent events, so multiply: 1/6 x 1/6 = 1/36
- `|`: at least one six: add, then subtract the overlap counted twice: 1/6 + 1/6 - 1/36

In [ ]:
a = rng.integers(1, 7, size=100000)
b = rng.integers(1, 7, size=100000)
print(((a == 6) & (b == 6)).mean(), 1/36)
print(((a == 6) | (b == 6)).mean(), 1/6 + 1/6 - 1/36)

## 2. Conditional probability
**P(A | B)** means *the probability of A, given that we know B happened*. Knowing B shrinks the world to only the cases where B is true.

### 1,000 emails
- `emails.csv`: 1,000 emails: does it contain 'free', 'winner', 'meeting' (1 = yes), and is it spam (1 = yes)
- `df.head()`: the first five emails

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "probability-and-bayes/emails.csv")
df = pd.read_csv(url)
print(df.shape)
df.head()

### Simple probabilities
- `df["is_spam"].mean()`: the share of 1s: the share of emails that are spam
- `p_free`: the share of emails containing 'free'

In [ ]:
p_spam = df["is_spam"].mean()
p_free = df["has_free"].mean()
print("P(spam) =", p_spam)
print("P(free) =", p_free)

### A count table
- `pd.crosstab(rows, columns)`: counts every combination: free yes/no against spam yes/no
- `margins=True`: adds the totals (All) for each row and column

In [ ]:
pd.crosstab(df["has_free"], df["is_spam"], margins=True)

### P(spam | free) and P(free | spam)
- `df[df["has_free"] == 1]`: shrink the world: keep only emails with 'free'
- `["is_spam"].mean()`: the share of those that are spam
- `the two answers differ`: P(A | B) and P(B | A) are different questions

In [ ]:
p_spam_given_free = df[df["has_free"] == 1]["is_spam"].mean()
p_free_given_spam = df[df["is_spam"] == 1]["has_free"].mean()
print("P(spam | free) =", round(p_spam_given_free, 3))
print("P(free | spam) =", round(p_free_given_spam, 3))

## 3. Bayes' theorem
**Bayes' theorem** flips a conditional probability around: P(A | B) = P(B | A) x P(A) / P(B). In words: posterior = likelihood x prior / evidence.

### Bayes' theorem, checked against the data
- `p_free_given_spam`: the likelihood: how often spam contains 'free'
- `p_spam`: the prior: how common spam is before reading the email
- `p_free`: the evidence: how common the word 'free' is overall
- `result`: Bayes gives exactly the answer we counted directly

In [ ]:
p_free_given_spam = df[df["is_spam"] == 1]["has_free"].mean()
posterior = p_free_given_spam * p_spam / p_free
print("Bayes:  ", round(posterior, 3))
print("Counted:", round(p_spam_given_free, 3))

### The base rate trap: a fraud alarm
- `the three inputs`: prior (fraud is rare), and how the detector behaves for fraud and for normal payments
- `p_alarm`: the evidence: alarms from real fraud PLUS false alarms from normal payments
- `p_fraud_if_alarm`: Bayes: only about 4.7% of alarms are real fraud, because fraud is so rare

In [ ]:
p_fraud = 0.001          # 1 in 1,000 payments is fraud
p_alarm_if_fraud = 0.99  # the detector catches 99% of fraud
p_alarm_if_ok = 0.02     # and wrongly flags 2% of normal payments

p_alarm = p_alarm_if_fraud * p_fraud + p_alarm_if_ok * (1 - p_fraud)
p_fraud_if_alarm = p_alarm_if_fraud * p_fraud / p_alarm
print("P(alarm) =", round(p_alarm, 4))
print("P(fraud | alarm) =", round(p_fraud_if_alarm, 3))

### Checking it with a simulation
- `1_000_000`: underscores just make big numbers easier to read
- `rng.random(n) < p_fraud`: True for about 1 in 1,000 payments
- `np.where(fraud, ..., ...)`: fraud payments raise an alarm 99% of the time, normal ones 2% of the time
- `fraud[alarm].mean()`: of all payments that raised an alarm, the share that were really fraud

In [ ]:
n = 1_000_000
fraud = rng.random(n) < p_fraud
alarm = np.where(fraud,
                 rng.random(n) < p_alarm_if_fraud,
                 rng.random(n) < p_alarm_if_ok)
print("alarms:", alarm.sum())
print("real fraud among alarms:", round(fraud[alarm].mean(), 3))

## 4. Naive Bayes: combining clues
With several clues, we multiply their likelihoods, assuming the clues are independent given the class. That assumption is why it's called **naive**.

### Two clues: 'free' and 'winner'
- `spam / ham`: split the emails: ham is the usual name for normal, wanted email
- `score_spam`: prior x likelihood of 'free' x likelihood of 'winner', for spam
- `score_spam / (score_spam + score_ham)`: turn the two scores into a probability that adds up to 1

In [ ]:
spam = df[df["is_spam"] == 1]
ham = df[df["is_spam"] == 0]
score_spam = p_spam * spam["has_free"].mean() * spam["has_winner"].mean()
score_ham = (1 - p_spam) * ham["has_free"].mean() * ham["has_winner"].mean()
print("Naive Bayes:", round(score_spam / (score_spam + score_ham), 3))

## 5. Mini project: a tiny spam filter
A spam filter built only from counts and Bayes' theorem.

### The filter
- `for w in words`: go through the three clue words
- `ps if email[w] else 1 - ps`: if the word is there, use P(word | spam); if not, use P(no word | spam)
- `s / (s + h)`: the probability the email is spam
- `two test emails`: 'free' + 'winner' scores very high; just 'meeting' scores very low

In [ ]:
words = ["has_free", "has_winner", "has_meeting"]

def spam_probability(email):
    s, h = p_spam, 1 - p_spam
    for w in words:
        ps, ph = spam[w].mean(), ham[w].mean()
        s *= ps if email[w] else 1 - ps
        h *= ph if email[w] else 1 - ph
    return s / (s + h)

email_a = {"has_free": 1, "has_winner": 1, "has_meeting": 0}
email_b = {"has_free": 0, "has_winner": 0, "has_meeting": 1}
print(round(spam_probability(email_a), 3))
print(round(spam_probability(email_b), 3))

### How often is it right?
- `df.apply(..., axis=1)`: run the function on every row (every email)
- `probs > 0.5`: call it spam if the probability is above one half
- `accuracy`: the share of emails where our guess matches the truth

*Checked on the same emails it learned from; a fair test uses new emails, which comes up when you train real models.*

In [ ]:
probs = df.apply(spam_probability, axis=1)
predicted = (probs > 0.5).astype(int)
print("Accuracy:", round((predicted == df["is_spam"]).mean(), 3))

## Try it yourself
1. Change the fraud rate to 1 in 100. How does the chance of fraud after an alarm change?
2. Score an email that contains only "meeting".
3. Simulate 3 dice: how often is the total exactly 18? Compare with (1/6)**3.